#### 1. Do high-value new applications that end up in a successful loan application tend to follow longer or more complex processes than those that don't?

In [ ]:
import os
import sys
import subprocess
from tqdm import tqdm

steps = [
    "Check core library versions",
    "Uninstall numpy, pandas, matplotlib",
    "Install compatible versions",
    "Check/install Plotly",
    "Clone LogView (if needed)",
    "Install LogView requirements",
    "Update sys.path"
]

def run_quiet(cmd):
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\nCommand failed: {' '.join(cmd)}")
        print("Please check your environment or run with output enabled for details.")
        raise e

progress = tqdm(steps, desc="Setting up environment", unit="step")

try:
    # Check core versions
    progress.update()
    import numpy as np
    import pandas as pd
    import matplotlib

    versions_ok = (
        np.__version__.startswith("1.24") and
        pd.__version__.startswith("2.1") and
        matplotlib.__version__.startswith("3.7")
    )

    # Reinstall if needed
    if not versions_ok:
        progress.update()
        run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "numpy", "pandas", "matplotlib"])
        progress.update()
        run_quiet([sys.executable, "-m", "pip", "install", "numpy==1.24.4", "pandas==2.1.4", "matplotlib==3.7.3"])
    else:
        progress.update(2)

    # Check/install plotly
    progress.update()
    try:
        import plotly
    except ImportError:
        run_quiet([sys.executable, "-m", "pip", "install", "plotly"])

    # Clone logview if needed
    progress.update()
    if not os.path.exists("logview"):
        run_quiet(["git", "clone", "https://github.com/fzerbato/logview.git"])

    # Install requirements
    progress.update()
    run_quiet([sys.executable, "-m", "pip", "install", "-r", "logview/requirements.txt"])

    # Add paths
    progress.update()
    cwd = os.getcwd()
    logview_path = os.path.abspath("logview")
    for path in [cwd, logview_path]:
        if path not in sys.path:
            sys.path.append(path)

    progress.close()
    print("Setup complete. Please restart the kernel and run all cells.")
except Exception:
    progress.close()
    print("Setup failed. See the error above.")

#### Import modules (will work after restart)

In [ ]:
try:
    import zipfile
    import plotly.io as pio
    import pandas as pd
    import pm4py
    from logview.utils import LogViewBuilder
    from logview.predicate import *
    from filter_visualization import query_exploration_icicle, query_breakdown_pie
    from pm4py.objects.log.importer.xes import importer as xes_importer
    print("All modules successfully imported.")
except Exception as e:
    print(f"Import failed: {e}")
    print("Please restart the kernel and rerun this cell.")

pio.renderers.default = 'notebook'

### Load and Prepare Event Log

Then load your event log and format it for analysis with PM4Py. The code below does this for the BPI Challenge 2017 dataset:

In [ ]:
# Load data

csv_file = "BPI_Challenge_2017.csv"
zip_file = "BPI_Challenge_2017.zip"

if not os.path.exists(csv_file):
    if os.path.exists(zip_file):
        print(f"Extracting {csv_file} from {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extract(csv_file)
    else:
        raise FileNotFoundError(f"Both '{csv_file}' and '{zip_file}' not found")
    
CASE_ID_COL = 'case'
TIMESTAMP_COL = 'time'
ACTIVITY_COL = 'event'

# Read full data
bpi_data = pd.read_csv(csv_file, sep=',', quotechar='"')
bpi_data.columns = bpi_data.columns.str.strip()
bpi_data[TIMESTAMP_COL] = pd.to_datetime(bpi_data[TIMESTAMP_COL], format='%Y/%m/%d %H:%M:%S.%f')

# Limit to 500 unique cases
unique_cases = bpi_data[CASE_ID_COL].unique()[:500]
bpi_data = bpi_data[bpi_data[CASE_ID_COL].isin(unique_cases)]

# Format for PM4Py
log = pm4py.format_dataframe(bpi_data, case_id=CASE_ID_COL, activity_key=ACTIVITY_COL, timestamp_key=TIMESTAMP_COL)

display(log)

In [ ]:
# Build LogView
log_view = LogViewBuilder.build_log_view(log)

# Step 1
query_1 = Query('HighValue', [GreaterThanConstant('RequestedAmount', 10000)])
rs_high_value, _ = log_view.evaluate_query('rs_HighValue', log, query_1)

# Step 2
query_2 = Query('NewApplications', [EqToConstant('ApplicationType', 'New credit')])
rs_new_apps, _ = log_view.evaluate_query('rs_NewApplications', rs_high_value, query_2)

# Step 3
query_3 = Query('Successful', [EqToConstant('event', 'A_Pending')])
rs_successful, _ = log_view.evaluate_query('rs_Successful', rs_new_apps, query_3)

# Show Summary
summary = log_view.get_summary()

In [ ]:
query_exploration_icicle('rs_Successful', log_view, metric='avg_case_duration_seconds', details=False)
query_breakdown_pie('rs_Successful', log_view, metric='avg_case_duration_seconds', details=False)

In [ ]:
query_exploration_icicle('rs_Successful', log_view, metric='avg_events_per_case', details=False)
query_breakdown_pie('rs_Successful', log_view, metric='avg_events_per_case', details=False)

In [ ]:
query_exploration_icicle('rs_Successful', log_view, metric='avg_time_between_events', details=False)
query_breakdown_pie('rs_Successful', log_view, metric='avg_time_between_events', details=False)